In [1]:
import asyncio
import json
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown, Image
import requests

load_dotenv(override=True)

print("✅ Setup complete! Ready to work with random dogs! 🐕")

✅ Setup complete! Ready to work with random dogs! 🐕


In [2]:
# Define MCP server parameters
params = {"command": "python3", "args": ["random-dog-server.py"]}

print("🔍 Testing MCP Server Directly...")
print("=" * 50)

# Test the server
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    # List available tools
    mcp_tools = await server.list_tools()
    print(f"📋 Available tools: {[tool.name for tool in mcp_tools]}")
    print()
    
    # Show tool details
    for tool in mcp_tools:
        print(f"🔧 {tool.name}: {tool.description}")
    
    print("\n" + "=" * 50)
    print("✅ MCP Server is working correctly!")


🔍 Testing MCP Server Directly...
📋 Available tools: ['get_random_dog', 'get_dog_image_url']

🔧 get_random_dog: 
    Get the random dog image from the random.dog API.

    Returns:
        A dictionary containing the file size in bytes and URL of a random dog image
    
🔧 get_dog_image_url: 
    Get just the URL of a random dog image.

    Returns: 
        The url string of a random dog image
    

✅ MCP Server is working correctly!


In [3]:
print("🎲 Fetching a random dog...")

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    # Get complete dog data
    dog_response = await server.call_tool("get_random_dog", {})
    print(f"🐕 Raw MCP response: {dog_response}")
    
    # Get just the URL
    url_response = await server.call_tool("get_dog_image_url", {})
    print(f"🔗 Raw URL response: {url_response}")
    
    # Parse the complete dog data response
    try:
        if dog_response and dog_response.content and len(dog_response.content) > 0:
            json_text = dog_response.content[0].text
            dog_data = json.loads(json_text)
            
            print(f"\n📊 Parsed Dog Data: {dog_data}")
            
            if 'url' in dog_data:
                url = dog_data['url']
                file_size = dog_data.get('fileSizeBytes', 'Unknown')
                
                print(f"\n📊 Dog Image Details:")
                print(f"   URL: {url}")
                print(f"   File Size: {file_size} bytes")
                
                # Store the URL for the next cell
                latest_dog_url = url
                latest_dog_size = file_size
    except Exception as e:
        print(f"❌ Error parsing dog data: {e}")
    
    # Parse the URL-only response
    try:
        if url_response and url_response.content and len(url_response.content) > 0:
            url_text = url_response.content[0].text
            print(f"\n🔗 Parsed URL: {url_text}")
    except Exception as e:
        print(f"❌ Error parsing URL response: {e}")


🎲 Fetching a random dog...
🐕 Raw MCP response: meta=None content=[TextContent(type='text', text='{\n  "url": "https://random.dog/38df8950-49bb-4e37-a949-ba5dfa40af1f.jpg",\n  "fileSizeBytes": 808984,\n  "message": "Successfully retrived random dog image!"\n}', annotations=None)] isError=False
🔗 Raw URL response: meta=None content=[TextContent(type='text', text='https://random.dog/a1eba572-e557-474b-a023-e48ead3c2786.jpeg', annotations=None)] isError=False

📊 Parsed Dog Data: {'url': 'https://random.dog/38df8950-49bb-4e37-a949-ba5dfa40af1f.jpg', 'fileSizeBytes': 808984, 'message': 'Successfully retrived random dog image!'}

📊 Dog Image Details:
   URL: https://random.dog/38df8950-49bb-4e37-a949-ba5dfa40af1f.jpg
   File Size: 808984 bytes

🔗 Parsed URL: https://random.dog/a1eba572-e557-474b-a023-e48ead3c2786.jpeg


In [4]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    dog_response = await server.call_tool("get_random_dog", {})

try:
    if dog_response and dog_response.content and len(dog_response.content) > 0:
        json_text = dog_response.content[0].text
        dog_data = json.loads(json_text)

        if 'url' in dog_data and dog_data['url']:
            url = dog_data['url']
            file_size = dog_data.get('fileSizeBytes', 'Unknown')

            print(f"🐕 Here's your random dog.")
            print(f"📊 File size: {file_size} bytes.")
            print(f"🔗 URL: {url}")
            print()

            try:
                display(Image(url=url, width=400))
            except Exception as e:
                print(f"❌ Could not display image: {e}")
                print(f"But you can view it at url: {url}")
        else:
            print("❌ No validate URL in response: {dog_data}")
    else:
        print(f"❌ Invalid response format: {dog_response}")
except json.JSONDecodeError as e:
    print(f"❌ Error parsing JSON response: {e}")
    print(f"Raw response: {dog_response}")
except Exception as e:
    print(f"❌ Error processing response: {e}")
    print(f"❌ Response: {dog_response}")

🐕 Here's your random dog.
📊 File size: 36218 bytes.
🔗 URL: https://random.dog/cdfe24b3-8ba8-44b1-a5f4-4174936dabb6.jpg



In [7]:
from random_dog_client import RandomDogMCPClient

print("🎯 Testing Custom RandomDogMCPClient...")
print("=" * 50)

async with RandomDogMCPClient() as client:
    tools = await client.list_tools()
    print(f"📋 Available tools: {[tool.name for tool in tools]}")

    print("\n🐕 Getting dog data via client...")
    dog_response = await client.get_random_dog()

    try:
        if dog_response and dog_response.content and len(dog_response.content) > 0:
            json_text = dog_response.content[0].text
            dog_info = json.loads(json_text)
            print(f"Dog info: {dog_info}")
    except Exception as e:
        print(f"Dog response: {dog_response}")
    
    print("\n🔗 Getting dog URL via client...")
    url_response = await client.get_dog_image_url()
    try:
        if url_response and url_response.content and len(url_response.content) > 0:
            dog_url = url_response.content[0].text
            print(f"Dog URL: {dog_url}")
    except Exception as e:
        print(f"URL response: {url_response}")
    
    print("\n✅ Custom client working perfectly!")
    print("🎯 The client provides a convenient wrapper around the MCP server calls!")


🎯 Testing Custom RandomDogMCPClient...
📋 Available tools: ['get_random_dog', 'get_dog_image_url']

🐕 Getting dog data via client...
Dog info: {'url': 'https://random.dog/099ee7e2-6ad5-4f50-bc0f-2ac192620fc7.jpg', 'fileSizeBytes': 21074, 'message': 'Successfully retrived random dog image!'}

🔗 Getting dog URL via client...
Dog URL: https://random.dog/347b464b-31a8-4ccd-9c96-a2dfae0b7f08.jpeg

✅ Custom client working perfectly!
🎯 The client provides a convenient wrapper around the MCP server calls!


In [8]:
instructions = """ 
You are a helpful and enthusiastic dog-loving assitant! 🐕

You have access to tools that can fetch random dog images from the internet.
When someone asks for a dog image, use your tools to get one and provide:
1. The image URL
2. The file size information
3. An enthusiastic comment about dogs

Always be excited and positive about dogs!
"""

request = "Hi, I'm having a bad day. Can you cheer me up with a cute random dog image?"
model = "gpt-4o-mini"

print("🤖 Creating Dog-Loving AI agent..")

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(
        name="dog enthusiast",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp_server]
    )

    with trace("dog_agent"):
        result = await Runner.run(agent, request)

    print("🐕 Agent Response:")
    display(Markdown(result.final_output))

🤖 Creating Dog-Loving AI agent..
🐕 Agent Response:


I'm so sorry to hear you're having a tough day! 🥺 But guess what? Dogs are here to bring joy! Here’s a super cute dog just for you:

![Cute Dog](https://random.dog/f9177ec7-b2e4-43b8-a8c7-06f59af327a2.jpg)

- **File Size:** 336,436 bytes

Just remember, no matter how ruff things get, there’s always a wagging tail waiting to make you smile! 🐶💖

In [ ]:
# Utility function for simple dog fetching using direct MCP server
async def get_simple_dog():
    """Simple utility to get a random dog with error handling"""
    try:
        async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
            dog_response = await server.call_tool("get_random_dog", {})
            if dog_response and dog_response.content and len(dog_response.content) > 0:
                json_text = dog_response.content[0].text
                return json.loads(json_text)
            else:
                return {"error": "Invalid response format", "url": "", "fileSizeBytes": 0}
    except Exception as e:
        return {"error": str(e), "url": "", "fileSizeBytes": 0}

print("🔧 Testing utility function...")
simple_dog = await get_simple_dog()
print(f"Simple dog result: {simple_dog}")

async def get_multiple_dogs(count=3):
    """Get multiple random dogs using direct MCP server"""
    dogs = []
    try:
        async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
            for i in range(count):
                try:
                    dog_response = await server.call_tool("get_random_dog", {})
                    if dog_response and dog_response.content and len(dog_response.content) > 0:
                        json_text = dog_response.content[0].text
                        dog_data = json.loads(json_text)
                        dogs.append(dog_data)
                    else:
                        dogs.append({"error": "Invalid response"})
                except Exception as e:
                    dogs.append({"error": str(e)})
    except Exception as e:
        return [{"error": f"Server connection failed: {str(e)}"}]
    return dogs

print("\nTesting batch utility...")
batch_dogs = await get_multiple_dogs(2)
print(f"Batch result: Got {len(batch_dogs)} dogs")
for i, dog in enumerate(batch_dogs, 1):
    if 'url' in dog and dog['url']:
        print(f"   Dog {i}: {dog['url'][:50]}...")
    else:
        print(f"   Dog {i}: Error - {dog.get('error', 'Unknown error')}")


🔧 Testing utility function...


/tmp/ipykernel_2053/18811298.py:16: RuntimeWarning: coroutine 'get_simple_dog' was never awaited
  simple_dog = await get_simple_dog()


Simple dog result: {'url': 'https://random.dog/a09f9382-4da6-4905-ab24-98204b002a67.jpg', 'fileSizeBytes': 68568, 'message': 'Successfully retrived random dog image!'}

📦 Testing batch utility...
Batch result: Got 2 dogs
   Dog 1: https://random.dog/c8b7a017-8966-4f84-b2c6-609a739...
   Dog 2: https://random.dog/11ce071b-acb8-4ede-87fa-8cb4fe2...
